# Final Decision: Deep Learning Model For Multi-Timeframe Trend Prediction

This notebook summarizes the latest saved outputs from all timeframe notebooks and provides a final model decision for deep learning deployment.

In [1]:
import pandas as pd

results = pd.DataFrame([
    {"timeframe": "15m", "best_model_in_notebook": "forest", "feature_group": "all_features", "accuracy": 0.6499, "balanced_accuracy": 0.5000, "f1_macro": 0.4717},
    {"timeframe": "1h",  "best_model_in_notebook": "forest", "feature_group": "all_features", "accuracy": 0.5210, "balanced_accuracy": 0.4745, "f1_macro": 0.4707},
    {"timeframe": "4h",  "best_model_in_notebook": "forest", "feature_group": "all_features", "accuracy": 0.4209, "balanced_accuracy": 0.4279, "f1_macro": 0.4027},
    {"timeframe": "1d",  "best_model_in_notebook": "forest", "feature_group": "price_action", "accuracy": 0.3986, "balanced_accuracy": 0.3917, "f1_macro": 0.3617},
    {"timeframe": "1w",  "best_model_in_notebook": "logistic", "feature_group": "indicators", "accuracy": 0.3423, "balanced_accuracy": 0.3944, "f1_macro": 0.2556},
    {"timeframe": "1MO", "best_model_in_notebook": "logistic", "feature_group": "all_features", "accuracy": 0.5882, "balanced_accuracy": 0.5038, "f1_macro": 0.4647},
])

results = results.sort_values("balanced_accuracy", ascending=False).reset_index(drop=True)
results

,timeframe,best_model_in_notebook,feature_group,accuracy,balanced_accuracy,f1_macro
0,1MO,logistic,all_features,0.5882,0.5038,0.4647
1,15m,forest,all_features,0.6499,0.5000,0.4717
2,1h,forest,all_features,0.5210,0.4745,0.4707
3,4h,forest,all_features,0.4209,0.4279,0.4027
4,1w,logistic,indicators,0.3423,0.3944,0.2556
5,1d,forest,price_action,0.3986,0.3917,0.3617


In [2]:
# Composite score prioritizes balanced performance over raw class-majority accuracy.
results["composite_score"] = (0.5 * results["balanced_accuracy"] + 0.3 * results["f1_macro"] + 0.2 * results["accuracy"])
summary = results.sort_values("composite_score", ascending=False).reset_index(drop=True)
summary

,timeframe,best_model_in_notebook,feature_group,accuracy,balanced_accuracy,f1_macro,composite_score
0,15m,forest,all_features,0.6499,0.5000,0.4717,0.52149
1,1MO,logistic,all_features,0.5882,0.5038,0.4647,0.50895
2,1h,forest,all_features,0.5210,0.4745,0.4707,0.48266
3,4h,forest,all_features,0.4209,0.4279,0.4027,0.41894
4,1d,forest,price_action,0.3986,0.3917,0.3617,0.38408
5,1w,logistic,indicators,0.3423,0.3944,0.2556,0.34234


## Finalized Decision

1. **Primary Deep Learning Model: Temporal Fusion Transformer (TFT)**
   - Why: Your best classical winners are mostly nonlinear and feature-rich (forest on 15m, 1h, 4h, 1d), which indicates interaction-heavy signal structure that sequence-aware attention models can capture better than linear methods.
   - TFT is suitable for multi-timeframe trend classification because it handles temporal dependencies, variable interactions, and regime shifts while remaining interpretable with variable importance over time.

2. **Training Setup (recommended for correct trend direction prediction)**
   - Task: 3-class direction prediction (`down`, `flat`, `up`) with the same horizon definition used in current notebooks.
   - Inputs: rolling windows of your existing engineered columns (`trend_pct_change`, support/resistance distances, indicators, candlestick features, volume features).
   - Window length: start with 64 bars for intraday (15m/1h), 48 bars for 4h/1d, and 24 bars for 1w/1MO.
   - Loss: class-weighted focal loss to improve minority-class recall.
   - Validation: walk-forward splits (same time-aware logic), compare against existing forest/logistic baseline for each timeframe.

3. **Success Criteria To Accept Deep Learning**
   - Balanced accuracy improvement of at least +0.03 over current notebook best per timeframe.
   - Macro F1 improvement of at least +0.03 over baseline.
   - Stable performance across at least 3 rolling out-of-sample folds.

4. **Risk Note**
   - Weekly and monthly datasets are smaller, so TFT may overfit unless strong regularization and early stopping are used.
   - If overfitting appears on 1w/1MO, use a smaller encoder or a shared model trained jointly on multiple timeframes with timeframe embedding.

### Bottom Line
Use **Temporal Fusion Transformer** as the single finalized deep learning model for deployment experiments, with forest as the control baseline. Promote TFT only if it beats current balanced-accuracy and macro-F1 thresholds on walk-forward validation.